# ColorCheck Reference Matcher Training

This notebook is the readable experiment entry point. Reusable implementation lives in `tools/ml/reference_matcher/`; the notebook does not hide dataset loading, model code, or evaluation logic in copied cells.

The experiment trains a compact PyTorch model to predict a 3D LUT from a target photograph and a different edited reference photograph. Real FiveK edits teach photographic judgment; exact synthetic shared-style pairs and a wrong-reference ranking objective teach reference responsiveness. All variants of a scene remain in one split.

In [ ]:
from pathlib import Path

DATASET_ZIP = Path('/Volumes/Untitled/archive (2).zip')
OUTPUT_DIR = Path('.local/reference-matcher')
assert DATASET_ZIP.exists(), f'Mount the SD card first: {DATASET_ZIP}'
DATASET_ZIP

## Smoke test

This intentionally small run proves that real images load, gradients update weights, validation executes, and checkpoints can be read. Its scores are not product-quality claims.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'tools.ml.reference_matcher.train',
    '--dataset-zip', str(DATASET_ZIP),
    '--output-dir', str(OUTPUT_DIR / 'smoke'),
    '--epochs', '1', '--batch-size', '8',
    '--image-size', '128', '--evaluation-image-size', '256',
    '--max-train-pairs', '32',
    '--max-validation-pairs', '16', '--max-test-pairs', '16',
], check=True)

## Measured pilot

Use Kaggle GPU timing to choose a run that fits the available quota. The initial pilot uses three epochs, balanced real/synthetic validation, and early stopping. The best checkpoint is retained independently of the final epoch.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'tools.ml.reference_matcher.train',
    '--dataset-zip', str(DATASET_ZIP),
    '--output-dir', str(OUTPUT_DIR), '--epochs', '3',
    '--batch-size', '32', '--image-size', '128',
    '--evaluation-image-size', '256', '--patience', '2',
], check=True)

## ONNX and calibrated INT8 export

Both FP32 and QDQ INT8 artifacts are exported for research. The manifest recommends no browser artifact unless the exact checkpoint passes held-out quality gates and INT8 remains within declared rendered-pixel tolerances.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'tools.ml.reference_matcher.export_onnx',
    '--checkpoint', str(OUTPUT_DIR / 'best.pt'),
    '--splits', str(OUTPUT_DIR / 'splits.json'),
    '--evaluation-summary', str(OUTPUT_DIR / 'summary.json'),
    '--dataset-zip', str(DATASET_ZIP),
], check=True)